# 🚀 AWS Weekend Challenge: Give Your Agent A Tool
## Project: CloudOps Sentinel – Equipping Amazon Bedrock with Live CloudWatch Telemetry & SLA Remediation
---
### 🎯 The Challenge
> *Take an agent you have already built and give it one new tool it did not have before, then show what it can now do.*

### 🏗️ Architecture & Overview
In large-scale enterprise environments (e.g. Healthcare Claims & Billing), hundreds of mission-critical data pipelines run across AWS on EMR, Glue, and S3.
- **Before (Baseline Agent)**: A conversational Bedrock AI assistant that knows architecture and best practices, but is **blind** to live runtime states and CloudWatch logs.
- **The One Tool Added**:  – directly wired via the **Amazon Bedrock Converse API ()** to probe live CloudWatch Logs Insights, EMR cluster runtimes, and S3 checkpoint metadata across 250 enterprise jobs.
- **After (Tool-Augmented Agent)**: Autonomously queries telemetry, extracts exact stack traces, detects SLA breaches, and outputs actionable root-cause analysis (RCA) with remediation commands.

In [1]:
import sys
import os
import json
from datetime import datetime

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.dirname(os.path.abspath("")) if "notebooks" in os.getcwd() else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root resolved: {PROJECT_ROOT}")

## 📦 Step 1: Inspecting the 250 Enterprise Production Jobs Inventory
Our telemetry tool is grounded in an official catalog of 250 healthcare/enterprise production pipelines (FACETS claims, billing, capitation, eligibility, etc.).

In [2]:
with open(os.path.join(PROJECT_ROOT, "official_250_jobs.json"), "r") as f:
    jobs = json.load(f)

print(f"Total monitored jobs loaded: {len(jobs)}")
print("Sample Jobs:")
for j in jobs[:10]:
    print(f"  - {j}")

## 🚫 Step 2: The Baseline Agent (BEFORE the Tool)
Let's see what happens when an on-call engineer asks the baseline Bedrock agent to diagnose a pipeline failure ().

In [3]:
from agent.baseline_agent import BaselineBedrockAgent

baseline_agent = BaselineBedrockAgent()
prompt = "Why did FACETS_CLAIMS_LOAD fail, what is our SLA breach status, and what should we do to fix it?"

print(f"User Inquiry: "{prompt}"
")
baseline_result = baseline_agent.ask(prompt)

print("--- BASELINE AGENT RESPONSE ---")
print(baseline_result["response"])

> 💡 **Observation**: The baseline agent admits it cannot inspect logs or runtime state, forcing engineers to manually comb through CloudWatch console streams.

## 🛠️ Step 3: Implementing "The One Tool" – AWS CloudWatch Telemetry Probe
We define the official tool specification conforming to the **Amazon Bedrock Converse API ()** format.

In [4]:
from tools.tool_spec import AWS_TELEMETRY_TOOL_SPEC
from tools.telemetry_probe import execute_telemetry_probe

print("--- Bedrock Converse API Tool Specification ---")
print(json.dumps(AWS_TELEMETRY_TOOL_SPEC, indent=2))

print("
--- Testing Direct Tool Execution for FACETS_CLAIMS_LOAD ---")
raw_telemetry = execute_telemetry_probe(job_name="FACETS_CLAIMS_LOAD", lookback_hours=24)
print(f"Job: {raw_telemetry['job_name']}")
print(f"Status: {raw_telemetry['status']}")
print(f"SLA Breached: {raw_telemetry['sla_breached']} (+{raw_telemetry['sla_breach_minutes']} mins)")
print(f"Failure Signature: {raw_telemetry['failure_signature']}")

## ⚡ Step 4: The Tool-Augmented Agent (AFTER the Tool)
Now, we ask the **Tool-Augmented Bedrock Agent** the exact same question.
Notice how Bedrock calls , extracts the live stack trace from CloudWatch, and formats an actionable SRE Incident Report.

In [5]:
from agent.bedrock_agent import ToolAugmentedBedrockAgent

augmented_agent = ToolAugmentedBedrockAgent()
augmented_result = augmented_agent.ask(prompt)

print("--- TOOL-AUGMENTED AGENT RESPONSE ---")
print(augmented_result["response"])

print("
--- Bedrock Tool Call Frame Inspection ---")
for call in augmented_result["tool_calls"]:
    print(f"Invoked Tool: {call['tool_name']}")
    print(f"Tool Input: {call['input']}")

## 📊 Step 5: Side-by-Side Comparison Matrix
Here is the stark contrast between the Agent Before and After receiving the tool.

In [6]:
import pandas as pd

comparison_data = {
    "Dimension": [
        "Live CloudWatch Visibility",
        "SLA Breach Calculation",
        "Root Cause Isolation",
        "Actionable Remediation",
        "On-Call Time Saved"
    ],
    "Before (Baseline Agent)": [
        "❌ None (Advises manual console search)",
        "❌ Unknown (Cannot read timestamps)",
        "❌ Generic guesses / Hallucination risk",
        "❌ Generic documentation links",
        "0 minutes (Engineer does all the work)"
    ],
    "After (With Telemetry Tool)": [
        "✅ Direct CloudWatch Log stream & cluster inspection",
        "✅ Exact duration vs limit & breach minutes delta",
        "✅ Pinpoints exact JVM heap/schema line & node IP",
        "✅ Exact copy-paste AWS CLI / EMR step command",
        "⚡ 30-45 minutes per critical incident"
    ]
}

df = pd.DataFrame(comparison_data)
from IPython.display import display, HTML
display(HTML(df.to_html(index=False)))